# Wikipedia RAG — eight-shard subset
Shards 00000–00007 are embedded: 1,250,310 articles and 5,665,798 passages. The remaining 33 shards are not included.
A background job builds direct cosine HNSW on the saved halfvec embeddings (m=16, ef_construction=128). Keep RUN_INGEST=False and BUILD_INDEX=False. Run the next cell for build status; it is a snapshot and can be rerun. Queries become available only after validation finishes.
The database schema retains its original name, rag_wiki_bge_full_v1, but the UI labels coverage as an eight-shard subset. This is not the full Wikipedia collection.


In [1]:
from pathlib import Path
import json, datetime
job_dir=Path('/Users/amolsingh/Documents/code/IITB-DL/data/pgvector_full_job')
status_path=job_dir/'index_status.json'
if status_path.exists():
    status=json.loads(status_path.read_text())
    print('Index phase:',status['phase'])
    print('Scope: eight-shard subset (00000–00007)')
    print('Articles:',format(status.get('articles',1250310),','))
    print('Passages:',format(status.get('chunks',5665798),','))
    print('Last update:',datetime.datetime.fromtimestamp(status['updated_at']).isoformat(timespec='seconds'))
    print('PostgreSQL progress:',status.get('build_progress','Not reported yet'))
    if status.get('message'): print(status['message'])
    if status.get('index_size'): print('Index size:',status['index_size'])
    print('Log:',job_dir/'index_build.log')
    print('Rerun this cell to refresh. Build counters are not a reliable completion-time estimate.')
else:
    print('Index status is not available yet.')


Index phase: complete
Scope: eight-shard subset (00000–00007)
Articles: 1,250,310
Passages: 5,665,798
Last update: 2026-10-27T02:10:37
PostgreSQL progress: {'phase': 'building index: loading tuples', 'blocks_total': 1297637, 'blocks_done': 1297637, 'tuples_total': 0, 'tuples_done': 5665793}
Index size: 6323 MB
Log: /Users/amolsingh/Documents/code/IITB-DL/data/pgvector_full_job/index_build.log
Rerun this cell to refresh. Build counters are not a reliable completion-time estimate.


In [2]:
from pathlib import Path
import os, sys, json, time, re, importlib.util, importlib.metadata
ROOT = Path('/Users/amolsingh/Documents/code/IITB-DL')
DATASET_DIR = ROOT / 'data/wikipedia_snapshot'
SQLITE_SOURCE = ROOT / 'data/full_wikipedia_bm25.sqlite'
MODEL_DIR = ROOT / 'models/flan-t5-small'
EMBED_DIR = ROOT / 'models/bge-small-en-v1.5'
MODEL_ID = 'BAAI/bge-small-en-v1.5'
MODEL_REVISION = '5c38ec7c405ec4b44b94cc5a9bb96e735b38267a'
PG_DSN = os.environ.get('RAG_PG_DSN','host=localhost dbname=rag_vectors')
SCOPE = 'full'  # Existing schema; actual coverage is the validated eight-shard subset.
RUN_INGEST = False
BUILD_INDEX = False
RUN_EVALUATION = False
ALLOW_DOWNLOADS = False
INDEX_KIND = 'half'  # Direct cosine HNSW on saved halfvec embeddings.
INGEST_DEVICE = 'mps'  # Apple GPU validated against CPU embeddings on this Mac.
QUERY_DEVICE = 'cpu'
ENCODE_BATCH_SIZE = 32
SOURCE_BATCH_SIZE = 32
QUERY_THREADS = 1
ANN_CANDIDATES = 100
EF_SEARCH = 200
DB_DISK_PATH = Path('/opt/homebrew/var/postgresql@14')  # Local server volume; change for a remote server.
SCHEMA = 'rag_wiki_bge_pilot_v1' if SCOPE=='pilot' else 'rag_wiki_bge_full_v1'
if SCOPE not in {'pilot','full'}:raise ValueError('SCOPE must be pilot or full')
# Credentials belong in RAG_PG_DSN / .pgpass, never in saved notebook outputs.
requirements={'psycopg':'psycopg[binary]==3.2.10','pgvector':'pgvector==0.4.1',
 'sentence_transformers':'sentence-transformers==6.2.0','pyarrow':'pyarrow==26.0.0',
 'ipywidgets':'ipywidgets==8.1.8'}
missing=[v for k,v in requirements.items() if importlib.util.find_spec(k) is None]
if missing:
    if not ALLOW_DOWNLOADS:raise RuntimeError('Install these in this kernel while online: '+', '.join(missing))
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
if not (EMBED_DIR/'model.safetensors').exists():
    if not ALLOW_DOWNLOADS:raise FileNotFoundError('BGE model missing. Enable ALLOW_DOWNLOADS once for setup.')
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL_ID,revision=MODEL_REVISION,local_dir=EMBED_DIR,
        allow_patterns=['*.json','*.safetensors','vocab.txt','1_Pooling/*','README.md'],ignore_patterns=['onnx/*','openvino/*'])
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
print('Collection:',SCHEMA,'scope:',SCOPE,'Python:',sys.executable)


Collection: rag_wiki_bge_full_v1 scope: full Python: /opt/homebrew/Caskroom/miniforge/base/bin/python


## Implementation
All implementation is included below; no helper Python file is needed. Source text is data, never executable instructions. No custom model code is trusted. Parameterized SQL and validated schema names separate queries from SQL syntax. The reader uses a persistent connection and read-only transactions; ingestion uses a separate connection and an advisory lock. For a multi-user service, replace this notebook connection pattern with a bounded pool and least-privilege database roles.

Chunking prefers sentence boundaries within 220 body tokens, overlaps 32 tokens, and includes up to 28 title tokens in the embedding. Overlong sentences are split rather than silently truncated. Original character offsets, article IDs and URLs are retained. No navigation text is silently deleted from the corpus. Model inputs exceeding their limits raise an error.


In [3]:
from pathlib import Path
from contextlib import contextmanager
from collections import OrderedDict
import os, re, json, time, hashlib, sqlite3, shutil, math, bisect
import numpy as np
import pyarrow.parquet as pq
import psycopg
from psycopg.rows import dict_row
from pgvector.psycopg import register_vector
from pgvector import HalfVector
from sentence_transformers import SentenceTransformer

QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
DIM = 384

def connect_pg(dsn):
    conn = psycopg.connect(dsn, autocommit=True, row_factory=dict_row, connect_timeout=5)
    register_vector(conn)
    return conn

def schema_name(name):
    if not re.fullmatch(r'rag_[a-z0-9_]{1,50}',name):
        raise ValueError('Schema must start rag_ and contain only lowercase letters, digits, underscores')
    return name

class Embedder:
    def __init__(self,path,model_id,revision,device='cpu',batch_size=64):
        import torch
        torch.set_num_threads(4)
        self.model=SentenceTransformer(str(path),device=device,local_files_only=True,trust_remote_code=False)
        self.model.max_seq_length=256
        self.tokenizer=self.model.tokenizer
        self.model_id,self.revision,self.batch_size=model_id,revision,batch_size
        self.prefix=QUERY_PREFIX if model_id=='BAAI/bge-small-en-v1.5' else ''
    def encode(self,texts,query=False):
        texts=[self.prefix+t if query else t for t in texts]
        lengths=[len(self.tokenizer.encode(t,add_special_tokens=True)) for t in texts]
        if any(n>256 for n in lengths):
            raise ValueError('Input exceeds 256 tokens; chunk or shorten it instead of silently truncating.')
        vectors=self.model.encode(texts,batch_size=self.batch_size,normalize_embeddings=True,
            convert_to_numpy=True,show_progress_bar=False).astype('float32')
        if vectors.shape!=(len(texts),DIM) or not np.isfinite(vectors).all():
            raise ValueError('Invalid embedding shape or nonfinite embedding')
        return vectors
    def chunk(self,article):
        # Prefer sentence endings within a 220-token body; retain exact source offsets.
        text=article['text']
        encoded=self.tokenizer(text,add_special_tokens=False,return_offsets_mapping=True,
                               truncation=False,verbose=False)
        offsets=encoded['offset_mapping'];n=len(offsets)
        title_ids=self.tokenizer.encode(article['title'],add_special_tokens=False)[:28]
        embed_title=self.tokenizer.decode(title_ids)
        sentence_ends=sorted(m.end() for m in re.finditer(r'[.!?](?:\s+|$)|\n\s*\n',text))
        start=0;ordinal=0
        while start<n:
            end=min(start+220,n)
            if end<n:
                choices=[j for j in range(start+110,end) if bisect.bisect_left(sentence_ends,offsets[j-1][1]) < bisect.bisect_right(sentence_ends,offsets[j][0])]
                if choices:end=choices[-1]
            char_start,char_end=offsets[start][0],offsets[end-1][1]
            body=text[char_start:char_end]
            embed_text=embed_title+'\n'+body
            # Re-tokenization at boundaries can differ from token slices.
            while len(self.tokenizer.encode(embed_text))>256 and end>start+1:
                end-=1;char_end=offsets[end-1][1];body=text[char_start:char_end];embed_text=embed_title+'\n'+body
            yield dict(chunk_id=f"{article['id']}:{char_start}:{char_end}",article_id=str(article['id']),
                title=article['title'],url=article['url'],text=body,char_start=char_start,char_end=char_end,
                embed_text=embed_text)
            if end==n:break
            start=max(start+1,end-32);ordinal+=1

def full_manifest(snapshot):
    files=sorted((Path(snapshot)/'20231101.en').glob('train-*.parquet'))
    if [p.name for p in files]!=[f'train-{i:05d}-of-00041.parquet' for i in range(41)]:
        raise RuntimeError('Full scope requires all 41 English shards')
    return dict(scope='full',dataset='wikimedia/wikipedia',snapshot='20231101.en',revision='b04c8d1ceb2f5cd4588862100d08de323dccfbaa',files=[dict(name=p.name,size=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns,
        articles=pq.ParquetFile(p).metadata.num_rows) for p in files])

def full_units(snapshot,batch_size=32,completed=None):
    completed=completed or {}
    done_shards={}
    for unit,articles in completed.items():
        shard=unit.split(':')[0];done_shards[shard]=done_shards.get(shard,0)+articles
    for path in sorted((Path(snapshot)/'20231101.en').glob('train-*.parquet')):
        parquet=pq.ParquetFile(path)
        if done_shards.get(path.name,0)==parquet.metadata.num_rows:continue
        for i,batch in enumerate(parquet.iter_batches(batch_size=batch_size,
                columns=['id','title','url','text'],use_threads=False)):
            unit=f'{path.name}:{i}'
            if unit not in completed:yield unit,batch.to_pylist()

class Store:
    def __init__(self,dsn,schema,config):
        self.schema=schema_name(schema);self.db=connect_pg(dsn);self.config=config
        s=self.schema
        with self.db.transaction():
            self.db.execute(f'CREATE SCHEMA IF NOT EXISTS {s}')
            self.db.execute(f'CREATE TABLE IF NOT EXISTS {s}.metadata (key text PRIMARY KEY,value jsonb NOT NULL)')
            old=self.db.execute(f"SELECT value FROM {s}.metadata WHERE key='config'").fetchone()
            if old and old['value']!=config:raise RuntimeError('Collection configuration changed. Choose a NEW schema; existing embeddings cannot be mixed.')
            if not old:self.db.execute(f'INSERT INTO {s}.metadata VALUES (%s,%s::jsonb)',('config',json.dumps(config)))
            self.db.execute(f'''CREATE TABLE IF NOT EXISTS {s}.chunks (
                id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,chunk_id text UNIQUE NOT NULL,
                article_id text NOT NULL,title text NOT NULL,url text NOT NULL,body text NOT NULL,
                char_start integer NOT NULL,char_end integer NOT NULL,embedding halfvec(384) NOT NULL)''')
            self.db.execute(f'CREATE TABLE IF NOT EXISTS {s}.checkpoints (unit text PRIMARY KEY,articles bigint,chunks bigint,seconds double precision)')
    def meta(self,key):
        row=self.db.execute(f'SELECT value FROM {self.schema}.metadata WHERE key=%s',(key,)).fetchone()
        return row['value'] if row else None
    def put(self,key,value):
        self.db.execute(f'INSERT INTO {self.schema}.metadata VALUES (%s,%s::jsonb) ON CONFLICT(key) DO UPDATE SET value=excluded.value',(key,json.dumps(value)))
    def ingest(self,units,embedder,expected_articles,min_free_gb=30,db_disk_path=None):
        s=self.schema
        # Session lock prevents two kernels ingesting the same collection concurrently.
        lock=self.db.execute('SELECT pg_try_advisory_lock(hashtext(%s)) AS locked',(s,)).fetchone()['locked']
        if not lock:raise RuntimeError('Another writer is using this collection')
        try:
            if self.meta('complete'):return self.meta('audit')
            completed={r['unit'] for r in self.db.execute(f'SELECT unit FROM {s}.checkpoints')}
            for unit,articles in units:
                if unit in completed:continue
                if db_disk_path and shutil.disk_usage(db_disk_path).free<min_free_gb*1e9:
                    raise RuntimeError('Low disk space; ingestion stopped safely. Free space and resume.')
                t=time.perf_counter();chunks=[p for a in articles for p in embedder.chunk(a)]
                # Embedding outside the transaction avoids holding DB locks during inference.
                vectors=embedder.encode([p['embed_text'] for p in chunks]) if chunks else []
                with self.db.transaction():
                    with self.db.cursor().copy(f'COPY {s}.chunks (chunk_id,article_id,title,url,body,char_start,char_end,embedding) FROM STDIN') as writer:
                        for p,v in zip(chunks,vectors):
                            writer.write_row((p['chunk_id'],p['article_id'],p['title'],p['url'],p['text'],p['char_start'],p['char_end'],HalfVector(v)))
                    self.db.execute(f'INSERT INTO {s}.checkpoints VALUES (%s,%s,%s,%s)',
                        (unit,len(articles),len(chunks),time.perf_counter()-t))
                print(unit,len(articles),'articles',len(chunks),'chunks',round(time.perf_counter()-t,1),'seconds',flush=True)
            audit=self.db.execute(f'SELECT coalesce(sum(articles),0)::bigint AS articles,coalesce(sum(chunks),0)::bigint AS chunks FROM {s}.checkpoints').fetchone()
            actual=self.db.execute(f'SELECT count(*) AS n FROM {s}.chunks').fetchone()['n']
            if audit['articles']!=expected_articles or audit['chunks']!=actual:
                raise RuntimeError('Coverage audit failed; collection is not marked complete')
            with self.db.transaction():
                self.put('audit',audit);self.put('complete',True)
            return audit
        finally:self.db.execute('SELECT pg_advisory_unlock(hashtext(%s))',(s,))
    def build_index(self,kind='binary',maintenance_mb=512):
        if kind not in {'binary','half'}:raise ValueError('Choose binary or half')
        if not self.meta('complete'):raise RuntimeError('Finish ingestion before indexing')
        s=self.schema;name='chunks_'+kind+'_hnsw'
        existing=self.db.execute('SELECT i.indisvalid FROM pg_index i JOIN pg_class c ON c.oid=i.indexrelid JOIN pg_namespace n ON n.oid=c.relnamespace WHERE n.nspname=%s AND c.relname=%s',(s,name)).fetchone()
        if existing and not existing['indisvalid']:raise RuntimeError('An invalid index exists; inspect it before rebuilding')
        self.db.execute("SELECT set_config('maintenance_work_mem',%s,false)",(f'{int(maintenance_mb)}MB',))
        expression='(binary_quantize(embedding)::bit(384)) bit_hamming_ops' if kind=='binary' else 'embedding halfvec_cosine_ops'
        # Offline initial build: regular CREATE INDEX is transactional and cannot leave
        # an invalid concurrently-built index. It blocks writes, not existing readers.
        self.db.execute(f'CREATE INDEX IF NOT EXISTS {name} ON {s}.chunks USING hnsw ({expression}) WITH (m=16,ef_construction=128)')
        self.db.execute(f'ANALYZE {s}.chunks');self.put('index_'+kind,True)
        return self.db.execute('SELECT pg_size_pretty(pg_relation_size(%s::regclass)) AS size',(s+'.'+name,)).fetchone()
    def close(self):self.db.close()

class VectorRetriever:
    def __init__(self,dsn,schema,embedder,kind='binary',candidates=100,ef_search=200,require_complete=True):
        self.schema=schema_name(schema);self.db=connect_pg(dsn);self.embedder=embedder
        self.kind,self.candidates,self.ef_search=kind,int(candidates),int(ef_search);self.cache=OrderedDict();self.last_timings={}
        if kind not in {'binary','half'}:raise ValueError('Invalid ANN index kind')
        if not 1<=self.candidates<=self.ef_search<=1000:raise ValueError('Require 1 <= candidates <= ef_search <= 1000')
        meta={r['key']:r['value'] for r in self.db.execute(f'SELECT * FROM {self.schema}.metadata')}
        if require_complete and not (meta.get('complete') or (meta.get('subset_ready') and meta.get('subset_audit'))):raise RuntimeError('Collection not ready: wait for the background index build and validation.')
        if not meta.get('index_'+kind):raise RuntimeError('Requested ANN index has not been built')
        if (meta['config']['model_id'],meta['config']['model_revision'])!=(embedder.model_id,embedder.revision):raise RuntimeError('Query and document embedding models differ')
        self.scope='eight-shard subset (00000–00007)' if meta.get('subset_ready') and not meta.get('complete') else meta['config']['source']['scope']
        self.db.execute("SET statement_timeout='10s'")
    def search(self,question,k=5,use_cache=True):
        if not question.strip() or not 1<=k<=20:raise ValueError('Nonempty question and k=1..20 required')
        key=(question.strip(),k,self.kind,self.candidates,self.ef_search)
        if use_cache and key in self.cache:
            import copy
            self.last_timings=dict(query_embedding_ms=0,ann_rerank_ms=0,fetch_ms=0,total_ms=0,cache_hit=True)
            return copy.deepcopy(self.cache[key])
        started=time.perf_counter();vec=HalfVector(self.embedder.encode([question],query=True)[0]);encoded=time.perf_counter()
        s=self.schema
        if self.kind=='binary':
            ordering='binary_quantize(embedding)::bit(384) <~> binary_quantize(%s::halfvec(384))::bit(384)'
        else:ordering='embedding <=> %s::halfvec(384)'
        with self.db.transaction():
            self.db.execute('SET TRANSACTION READ ONLY')
            self.db.execute("SELECT set_config('hnsw.ef_search',%s,true)",(str(self.ef_search),))
            # No global sort: ANN shortlist first, exact cosine rerank only that shortlist.
            rows=self.db.execute(f'''WITH candidates AS MATERIALIZED (
                SELECT id,embedding FROM {s}.chunks ORDER BY {ordering} LIMIT %s)
                SELECT id,embedding <=> %s::halfvec(384) AS distance FROM candidates
                ORDER BY distance,id LIMIT %s''',(vec,self.candidates,vec,k)).fetchall()
            searched=time.perf_counter()
            ids=[r['id'] for r in rows]
            fetched=self.db.execute(f'SELECT id,chunk_id AS passage_id,article_id,title,url,body AS text,char_start,char_end FROM {s}.chunks WHERE id=ANY(%s)',(ids,)).fetchall() if ids else []
        byid={r['id']:r for r in fetched};hits=[dict(byid[r['id']],distance=float(r['distance']),similarity=1-float(r['distance'])) for r in rows]
        ended=time.perf_counter();self.last_timings=dict(query_embedding_ms=(encoded-started)*1000,ann_rerank_ms=(searched-encoded)*1000,fetch_ms=(ended-searched)*1000,total_ms=(ended-started)*1000,cache_hit=False)
        if use_cache:
            import copy
            self.cache[key]=copy.deepcopy(hits)
            if len(self.cache)>128:self.cache.popitem(last=False)
        return hits
    def explain(self,question):
        v=HalfVector(self.embedder.encode([question],query=True)[0])
        expr='binary_quantize(embedding)::bit(384) <~> binary_quantize(%s::halfvec(384))::bit(384)' if self.kind=='binary' else 'embedding <=> %s::halfvec(384)'
        return self.db.execute(f'EXPLAIN (ANALYZE,BUFFERS,FORMAT JSON) SELECT id FROM {self.schema}.chunks ORDER BY {expr} LIMIT %s',(v,self.candidates)).fetchone()['QUERY PLAN']
    def close(self):self.db.close()


W1027 08:51:24.352000 71219 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1027 08:51:24.719000 71219 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1027 08:51:24.742000 71219 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [4]:
PILOT_CONFIG = {'model_id': 'BAAI/bge-small-en-v1.5', 'model_revision': '5c38ec7c405ec4b44b94cc5a9bb96e735b38267a', 'dim': 384, 'normalized': True, 'storage': 'halfvec', 'chunker': {'version': 1, 'body_tokens': 220, 'overlap_tokens': 32, 'title_tokens': 28}, 'source': {'scope': 'pilot', 'articles': 65, 'sha256': 'ddadac28c7cd069b297d24b7834c43b8cee2aac0afd88a1fd0649f138eba2667'}}
PILOT_ARTICLE_ORDER = ['12', '5275598', '6588983', '8125088', '11436530', '11825851', '13194570', '14993481', '16794303', '18634645', '20248062', '21921237', '23583319', '25196054', '27014469', '28819981', '30872633', '31910502', '33765855', '35786469', '37433802', '39288352', '41013059', '43619652', '45516682', '47761388', '51350094', '53434821', '55614901', '57373928', '59666823', '62454053', '64504165', '66871507', '68905044', '71273793', '73886145', '379233', '1300584', '2095464', '2983210', '586', '339', '746', '600', '303', '628', '904', '620', '662', '738', '680', '358', '851', '802', '775', '375986', '39', '308', '852', '624', '844', '736', '51585', '698']
DEVELOPMENT_QUESTIONS = [{'id': 'q01', 'question': 'What is the atomic number of aluminium?', 'answerable': True, 'answers': ['13'], 'article_id': '904', 'source_title': 'Aluminium', 'source_url': 'https://en.wikipedia.org/wiki/Aluminium', 'evidence_quote': 'is a chemical element with the symbol\xa0Al and atomic number\xa013', 'gold_passage_ids': ['904:0']}, {'id': 'q02', 'question': 'What is the capital of Alabama?', 'answerable': True, 'answers': ['Montgomery'], 'article_id': '303', 'source_title': 'Alabama', 'source_url': 'https://en.wikipedia.org/wiki/Alabama', 'evidence_quote': "Alabama's capital is Montgomery", 'gold_passage_ids': ['303:0', '303:1']}, {'id': 'q03', 'question': 'In which city was Aristotle born?', 'answerable': True, 'answers': ['Stagira'], 'article_id': '308', 'source_title': 'Aristotle', 'source_url': 'https://en.wikipedia.org/wiki/Aristotle', 'evidence_quote': 'He was born in the city of Stagira in northern Greece during the Classical period', 'gold_passage_ids': ['308:0']}, {'id': 'q04', 'question': 'What philosophical system did Ayn Rand develop?', 'answerable': True, 'answers': ['Objectivism'], 'article_id': '339', 'source_title': 'Ayn Rand', 'source_url': 'https://en.wikipedia.org/wiki/Ayn%20Rand', 'evidence_quote': 'developing a philosophical system she named Objectivism', 'gold_passage_ids': ['339:0']}, {'id': 'q05', 'question': 'What is the capital of Algeria?', 'answerable': True, 'answers': ['Algiers'], 'article_id': '358', 'source_title': 'Algeria', 'source_url': 'https://en.wikipedia.org/wiki/Algeria', 'evidence_quote': 'The capital and largest city is Algiers', 'gold_passage_ids': ['358:1']}, {'id': 'q06', 'question': 'How many code points does ASCII have?', 'answerable': True, 'answers': ['128'], 'article_id': '586', 'source_title': 'ASCII', 'source_url': 'https://en.wikipedia.org/wiki/ASCII', 'evidence_quote': 'ASCII has just 128 code points', 'gold_passage_ids': ['586:0']}, {'id': 'q07', 'question': 'What is the official language of Andorra?', 'answerable': True, 'answers': ['Catalan'], 'article_id': '600', 'source_title': 'Andorra', 'source_url': 'https://en.wikipedia.org/wiki/Andorra', 'evidence_quote': 'The official language is Catalan', 'gold_passage_ids': ['600:1', '600:2']}, {'id': 'q08', 'question': 'Who wrote Animal Farm?', 'answerable': True, 'answers': ['George Orwell', 'Orwell'], 'article_id': '620', 'source_title': 'Animal Farm', 'source_url': 'https://en.wikipedia.org/wiki/Animal%20Farm', 'evidence_quote': 'by George Orwell, first published in England on 17 August 1945', 'gold_passage_ids': ['620:0']}, {'id': 'q09', 'question': 'What is the state capital of Alaska?', 'answerable': True, 'answers': ['Juneau'], 'article_id': '624', 'source_title': 'Alaska', 'source_url': 'https://en.wikipedia.org/wiki/Alaska', 'evidence_quote': 'The state capital of Juneau is the second-largest city in the United States by area', 'gold_passage_ids': ['624:1', '624:2']}, {'id': 'q10', 'question': 'In which English town was Aldous Huxley born?', 'answerable': True, 'answers': ['Godalming'], 'article_id': '628', 'source_title': 'Aldous Huxley', 'source_url': 'https://en.wikipedia.org/wiki/Aldous%20Huxley', 'evidence_quote': 'Huxley was born in Godalming, Surrey, England, in 1894', 'gold_passage_ids': ['628:2']}, {'id': 'q11', 'question': 'Who piloted the Command Module Columbia during Apollo 11?', 'answerable': True, 'answers': ['Michael Collins', 'Collins'], 'article_id': '662', 'source_title': 'Apollo 11', 'source_url': 'https://en.wikipedia.org/wiki/Apollo%2011', 'evidence_quote': 'pilot Michael Collins flew the Command Module Columbia in lunar orbit', 'gold_passage_ids': ['662:1']}, {'id': 'q12', 'question': "Approximately what percentage of Earth's surface does the Atlantic Ocean cover?", 'answerable': True, 'answers': ['17%', '17 percent', '17'], 'article_id': '698', 'source_title': 'Atlantic Ocean', 'source_url': 'https://en.wikipedia.org/wiki/Atlantic%20Ocean', 'evidence_quote': "It covers approximately 17% of Earth's surface", 'gold_passage_ids': ['698:0']}, {'id': 'q13', 'question': 'In which year did Albert Einstein receive the Nobel Prize in Physics?', 'answerable': True, 'answers': ['1921'], 'article_id': '736', 'source_title': 'Albert Einstein', 'source_url': 'https://en.wikipedia.org/wiki/Albert%20Einstein', 'evidence_quote': 'He received the 1921 Nobel Prize in Physics', 'gold_passage_ids': ['736:1']}, {'id': 'q14', 'question': 'What is the capital of Albania?', 'answerable': True, 'answers': ['Tirana'], 'article_id': '738', 'source_title': 'Albania', 'source_url': 'https://en.wikipedia.org/wiki/Albania', 'evidence_quote': 'Tirana is the capital and largest city in the country', 'gold_passage_ids': ['738:1']}, {'id': 'q15', 'question': 'What is the capital of Azerbaijan?', 'answerable': True, 'answers': ['Baku'], 'article_id': '746', 'source_title': 'Azerbaijan', 'source_url': 'https://en.wikipedia.org/wiki/Azerbaijan', 'evidence_quote': 'Baku is the capital and largest city', 'gold_passage_ids': ['746:0']}, {'id': 'q16', 'question': 'On what date did Ankara become the new Turkish capital?', 'answerable': True, 'answers': ['29 October 1923', 'October 29, 1923'], 'article_id': '802', 'source_title': 'Ankara', 'source_url': 'https://en.wikipedia.org/wiki/Ankara', 'evidence_quote': 'Ankara became the new Turkish capital upon the establishment of the Republic on 29 October 1923', 'gold_passage_ids': ['802:2', '802:3']}, {'id': 'q17', 'question': 'In which Dutch province is Amsterdam located?', 'answerable': True, 'answers': ['North Holland'], 'article_id': '844', 'source_title': 'Amsterdam', 'source_url': 'https://en.wikipedia.org/wiki/Amsterdam', 'evidence_quote': 'Located in the Dutch province of North Holland', 'gold_passage_ids': ['844:0']}, {'id': 'q18', 'question': 'What explosive is Alfred Nobel most famous for inventing?', 'answerable': True, 'answers': ['dynamite'], 'article_id': '851', 'source_title': 'Alfred Nobel', 'source_url': 'https://en.wikipedia.org/wiki/Alfred%20Nobel', 'evidence_quote': "Nobel's most famous invention was dynamite", 'gold_passage_ids': ['851:0']}, {'id': 'q19', 'question': 'In which city was Alexander Graham Bell born?', 'answerable': True, 'answers': ['Edinburgh'], 'article_id': '852', 'source_title': 'Alexander Graham Bell', 'source_url': 'https://en.wikipedia.org/wiki/Alexander%20Graham%20Bell', 'evidence_quote': 'Bell was born in Edinburgh, Scotland, on March 3, 1847', 'gold_passage_ids': ['852:2']}, {'id': 'q20', 'question': 'To which continent is the aardvark native?', 'answerable': True, 'answers': ['Africa'], 'article_id': '680', 'source_title': 'Aardvark', 'source_url': 'https://en.wikipedia.org/wiki/Aardvark', 'evidence_quote': 'nocturnal mammal native to Africa', 'gold_passage_ids': ['680:0']}, {'id': 'q21', 'question': "What does Euclid's algorithm compute for two integers?", 'answerable': True, 'answers': ['greatest common divisor', 'greatest common divisor (GCD)', 'GCD'], 'article_id': '775', 'source_title': 'Algorithm', 'source_url': 'https://en.wikipedia.org/wiki/Algorithm', 'evidence_quote': 'an efficient method for computing the greatest common divisor (GCD) of two integers', 'gold_passage_ids': ['775:41']}, {'id': 'q22', 'question': 'What type of rocket launched Apollo 11?', 'answerable': True, 'answers': ['Saturn V', 'Saturn V rocket'], 'article_id': '662', 'source_title': 'Apollo 11', 'source_url': 'https://en.wikipedia.org/wiki/Apollo%2011', 'evidence_quote': 'Apollo 11 was launched by a Saturn V rocket', 'gold_passage_ids': ['662:1']}, {'id': 'q23', 'question': "In which year was Alfred Nobel's dynamite patented?", 'answerable': True, 'answers': ['1867'], 'article_id': '851', 'source_title': 'Alfred Nobel', 'source_url': 'https://en.wikipedia.org/wiki/Alfred%20Nobel', 'evidence_quote': 'dynamite, an explosive using  nitroglycerin; it was patented in 1867', 'gold_passage_ids': ['851:0']}, {'id': 'q24', 'question': 'What albedo value corresponds to a body that reflects all incident radiation?', 'answerable': True, 'answers': ['1', 'one'], 'article_id': '39', 'source_title': 'Albedo', 'source_url': 'https://en.wikipedia.org/wiki/Albedo', 'evidence_quote': '1 (corresponding to a body that reflects all incident radiation', 'gold_passage_ids': ['39:0']}, {'id': 'q25', 'question': 'Who won the Nobel Prize in Physics in 2024?', 'answerable': False, 'answers': [], 'article_id': None, 'source_title': None, 'source_url': None, 'evidence_quote': None, 'gold_passage_ids': [], 'unanswerable_reason': 'The question asks about an event after the November 2023 corpus snapshot.'}, {'id': 'q26', 'question': 'Which film won the Academy Award for Best Picture in 2024?', 'answerable': False, 'answers': [], 'article_id': None, 'source_title': None, 'source_url': None, 'evidence_quote': None, 'gold_passage_ids': [], 'unanswerable_reason': 'The question asks about an event after the November 2023 corpus snapshot.'}, {'id': 'q27', 'question': "Who won the Wimbledon men's singles title in 2024?", 'answerable': False, 'answers': [], 'article_id': None, 'source_title': None, 'source_url': None, 'evidence_quote': None, 'gold_passage_ids': [], 'unanswerable_reason': 'The question asks about an event after the November 2023 corpus snapshot.'}, {'id': 'q28', 'question': "What was Apple Inc.'s total revenue for fiscal year 2025?", 'answerable': False, 'answers': [], 'article_id': None, 'source_title': None, 'source_url': None, 'evidence_quote': None, 'gold_passage_ids': [], 'unanswerable_reason': 'The question asks about an event after the November 2023 corpus snapshot.'}]


In [5]:
def prepare_pilot_articles():
    # Reconstruct the same documented pilot from local data; never downloaded examples.
    articles={}
    for path in sorted((DATASET_DIR/'20231101.en').glob('train-*.parquet')):
        a=next(pq.ParquetFile(path).iter_batches(batch_size=1,columns=['id','title','url','text'])).to_pylist()[0]
        articles[str(a['id'])]=a
    conn=sqlite3.connect(SQLITE_SOURCE.as_uri()+'?mode=ro',uri=True)
    conn.row_factory=sqlite3.Row
    try:
        # These explicit article IDs cover the inherited development questions and two regressions.
        forced={q['article_id'] for q in DEVELOPMENT_QUESTIONS if q['answerable']}|{'51585','375986'}
        for aid in forced:
            rows=[dict(r) for r in conn.execute('SELECT * FROM passages WHERE passage_id>=? AND passage_id<?',(aid+':',aid+';'))]
            rows.sort(key=lambda r:r['char_start'])
            if not rows:raise RuntimeError('Pilot source article missing: '+aid)
            text=''
            for row in rows:
                if row['char_start']>len(text):text+=' '*(row['char_start']-len(text))
                if row['char_end']>len(text):text+=row['text'][max(0,len(text)-row['char_start']):]
            articles[aid]=dict(id=aid,title=rows[0]['title'],url=rows[0]['url'],text=text)
    finally:conn.close()
    ordered=[articles[aid] for aid in PILOT_ARTICLE_ORDER]
    if hashlib.sha256(json.dumps(ordered).encode()).hexdigest()!=PILOT_CONFIG['source']['sha256']:
        raise RuntimeError('Pilot source changed; use a new collection configuration instead of mixing embeddings.')
    return ordered

source = PILOT_CONFIG['source'] if SCOPE=='pilot' else full_manifest(DATASET_DIR)
config = dict(model_id=MODEL_ID,model_revision=MODEL_REVISION,dim=384,normalized=True,storage='halfvec',
    chunker=dict(version=1,body_tokens=220,overlap_tokens=32,title_tokens=28),source=source)
if SCOPE=='full':config['source_batch_size']=SOURCE_BATCH_SIZE
expected_articles = source['articles'] if SCOPE=='pilot' else sum(f['articles'] for f in source['files'])
if 'store' in globals():store.close()
store=Store(PG_DSN,SCHEMA,config)
print('Full-snapshot articles:',expected_articles,'Full snapshot complete:',bool(store.meta('complete')))
print('Subset coverage:',store.meta('subset_audit'),'Subset query-ready:',bool(store.meta('subset_ready')))
print('Stored coverage:',store.meta('audit'))


Full-snapshot articles: 6407814 Full snapshot complete: False
Subset coverage: {'scope': 'eight_shard_subset', 'chunks': 5665798, 'shards': ['train-00000-of-00041.parquet', 'train-00001-of-00041.parquet', 'train-00002-of-00041.parquet', 'train-00003-of-00041.parquet', 'train-00004-of-00041.parquet', 'train-00005-of-00041.parquet', 'train-00006-of-00041.parquet', 'train-00007-of-00041.parquet'], 'articles': 1250310, 'full_corpus_complete': False} Subset query-ready: True
Stored coverage: None


## Ingest once, resume safely
Each transaction writes passage rows and its checkpoint together. A crash before commit leaves neither; rerunning skips committed units. The collection configuration rejects changed models, sources, or chunk settings. Full completion is declared only after article-count and row-count audits. Empty articles count toward source coverage but yield no passages. Do not point a full run at the pilot schema.

Embedding uses CPU threads suited to batches; interactive querying uses one CPU thread, measured faster for short queries on this Mac. Keep the computer awake during a long run. A free-space check stops ingestion below 30 GB free on the specified local database volume. It cannot predict all WAL/index growth; monitor PostgreSQL and disk capacity during the full build. Once a full collection is complete, do not append to it from outside this notebook: query caches assume an immutable collection.


In [6]:
if RUN_INGEST and store.meta('subset_audit'):
    raise RuntimeError('This collection is a frozen eight-shard subset. Plan ingestion resumption separately before changing it.')
if RUN_INGEST:
    import torch
    writer_embedder=Embedder(EMBED_DIR,MODEL_ID,MODEL_REVISION,device=INGEST_DEVICE,batch_size=ENCODE_BATCH_SIZE)
    if SCOPE=='pilot':
        pilot_articles=prepare_pilot_articles()
        units=((f'pilot:{i}',pilot_articles[i:i+8]) for i in range(0,len(pilot_articles),8))
    else:
        done={r['unit']:r['articles'] for r in store.db.execute(f'SELECT unit,articles FROM {SCHEMA}.checkpoints')}
        units=full_units(DATASET_DIR,SOURCE_BATCH_SIZE,done)
    started=time.perf_counter()
    audit=store.ingest(units,writer_embedder,expected_articles,db_disk_path=DB_DISK_PATH)
    print('Coverage audit:',audit,'ingestion seconds this run:',time.perf_counter()-started)
    del writer_embedder
else:
    print('Ingestion skipped. Existing collection:',store.meta('audit'))


Ingestion skipped. Existing collection: None


In [7]:
progress=store.db.execute(f'SELECT coalesce(sum(articles),0) AS articles,coalesce(sum(chunks),0) AS chunks,coalesce(sum(seconds),0) AS seconds FROM {SCHEMA}.checkpoints').fetchone()
print('Progress:',progress)
print('Disk free GB:',round(shutil.disk_usage(DB_DISK_PATH).free/1e9,1))
print('Database collection size:',store.db.execute('SELECT pg_size_pretty(pg_total_relation_size(%s::regclass)) AS size',(SCHEMA+'.chunks',)).fetchone()['size'])
# Live index progress can also be inspected from a second SQL session:
# SELECT phase, blocks_done, blocks_total FROM pg_stat_progress_create_index;


Progress: {'articles': Decimal('1250310'), 'chunks': Decimal('5665798'), 'seconds': 73884.71886767358}
Disk free GB: 228.4
Database collection size: 17 GB


## Build the ANN index
The default binary HNSW index stores compact sign-bit signatures. Search retrieves 100 candidates with `ef_search=200`, then reranks them by cosine distance using the stored half vectors. The original half vectors remain available for exact evaluation. HNSW uses m=16 and ef_construction=128. Raising candidate count/ef_search can improve recall at a latency cost; benchmark on your own questions. The optional half-vector HNSW index uses more disk and memory. Do not blindly build both at full scale.

Initial indexes are built after bulk loading. Regular CREATE INDEX is used for this offline build; it blocks writes while building. A server crash rolls back an unfinished build, so ingestion can resume but the index build must restart. An explicit ANALYZE updates planner statistics. The following EXPLAIN check verifies the intended HNSW path rather than assuming an index is used.


In [8]:
if BUILD_INDEX and store.meta('subset_audit'):
    raise RuntimeError('The subset index is managed by the background build. Keep BUILD_INDEX=False.')
if BUILD_INDEX:
    print(store.build_index(INDEX_KIND,maintenance_mb=512))
else:
    print('Index build skipped; ready:',store.meta('index_'+INDEX_KIND))
import torch
query_embedder=Embedder(EMBED_DIR,MODEL_ID,MODEL_REVISION,device=QUERY_DEVICE,batch_size=ENCODE_BATCH_SIZE)
torch.set_num_threads(QUERY_THREADS)
if 'retriever' in globals():retriever.close()
retriever=VectorRetriever(PG_DSN,SCHEMA,query_embedder,kind=INDEX_KIND,candidates=ANN_CANDIDATES,ef_search=EF_SEARCH)
plan=retriever.explain('What is the capital of India?')
def index_nodes(value):
    if isinstance(value,dict):
        if 'Index Name' in value:yield value['Index Name']
        for child in value.values():yield from index_nodes(child)
    elif isinstance(value,list):
        for child in value:yield from index_nodes(child)
indexes=list(index_nodes(plan))
print('Query plan indexes:',indexes)
if 'chunks_'+INDEX_KIND+'_hnsw' not in indexes:
    raise RuntimeError('Planner did not choose the intended HNSW index. Inspect plan and ANALYZE before benchmarking.')


Index build skipped; ready: True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Query plan indexes: ['chunks_half_hnsw']


## Grounded answers and interactive UI
Flan-T5-small is retained from the original project. This is the main answer-quality constraint: a fast retriever does not turn a small generator into a production-quality assistant. Deterministic generation uses only budgeted evidence, returns source excerpts, and rejects answers absent from the visible passages. These lexical checks are not semantic entailment verification and can reject correct paraphrases or accept wrong relations. High-stakes or public deployment needs a stronger evaluated generator, independent unanswerable tests, calibrated rejection thresholds and human review.

The session's chat few-shot format is not copied verbatim into Flan-T5: it is a sequence-to-sequence model with a smaller context budget. Instead the compatible prompt uses short factual answers and abstention instructions. Both concatenate and generate-from-evidence are available. The latter makes more model calls. Citations are attached from verified source spans rather than allowing the model to invent IDs.

A fast sentence selection stage favors question terms and penalizes historical facts for present-tense questions and navigation text. Count questions favor sentences connecting a number with the requested noun. This compression is heuristic; it can miss paraphrases and complex temporal intent. Raw vector hits and selected context are both retained in the result for audit. It has a separate timing field.


In [9]:
def offline_mode() -> None:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [10]:
UNKNOWN = "I don't know based on the provided evidence."
WORD = re.compile(r"\w+", re.UNICODE)
STOPWORDS = set(
    "a an the and or but of to in on at for from by with as is are was were be "
    "been being do does did has have had what which who whom whose when where "
    "why how can could would should will may might that this these those it its "
    "into about according many much name tell me give".split()
)


In [11]:
def span(text: str, candidate: str) -> str | None:
    """Return an original-text span, matching complete casefolded word tokens."""
    needles = [m.group().casefold() for m in WORD.finditer(candidate)]
    if not needles:
        return None
    matches = list(WORD.finditer(text))
    haystack = [m.group().casefold() for m in matches]
    for i in range(len(haystack) - len(needles) + 1):
        if haystack[i : i + len(needles)] == needles:
            return text[matches[i].start() : matches[i + len(needles) - 1].end()]
    return None

def is_unknown(text: str) -> bool:
    normalized = " ".join(WORD.findall(text.casefold()))
    return not normalized or normalized in {
        "unknown", "unanswerable", "not enough information", "not provided",
        "i don t know", "i do not know", "no answer", "not in the context",
        "i don t know based on the provided evidence",
    }

def passages(article: dict, chunk_words: int = 100, overlap: int = 25):
    if chunk_words < 1 or overlap < 0 or overlap >= chunk_words:
        raise ValueError("Require chunk_words > overlap >= 0.")
    text = article["text"]
    words = list(re.finditer(r"\S+", text))
    for ordinal, start in enumerate(range(0, len(words), chunk_words - overlap)):
        end = min(start + chunk_words, len(words))
        char_start, char_end = words[start].start(), words[end - 1].end()
        yield {
            "passage_id": f'{article["id"]}:{ordinal}',
            "article_id": str(article["id"]),
            "title": article["title"],
            "url": article["url"],
            "text": text[char_start:char_end],
            "char_start": char_start,
            "char_end": char_end,
        }
        if end == len(words):
            break


In [12]:
class Generator:
    def __init__(self, model_dir: Path | None = None, device="cpu", threads=4,
                 max_input_tokens=512, max_new_tokens=24):
        offline_mode()
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        model_dir = model_dir or ROOT / "models" / "flan-t5-small"
        if not (model_dir / "model.safetensors").exists():
            raise FileNotFoundError(f"Model missing at {model_dir}. Run prepare while online.")
        if threads < 1 or max_new_tokens < 1 or not 64 <= max_input_tokens <= 512:
            raise ValueError("Require positive threads/output tokens and input budget 64..512.")
        torch.set_num_threads(threads)
        torch.manual_seed(42)
        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_dir, local_files_only=True, use_safetensors=True,
        ).to(device).eval()
        self.device = device
        self.max_input_tokens = max_input_tokens
        self.max_new_tokens = max_new_tokens
        self.threads = threads

    def count_tokens(self, text: str) -> int:
        return len(self.tokenizer.encode(text, add_special_tokens=True, verbose=False))

    @staticmethod
    def render(question: str, evidence: list[dict], stage="answer") -> str:
        if stage == "closed_book":
            return f"Answer the question briefly.\nQuestion: {question}\nAnswer:"
        instruction = (
            "Copy the shortest phrase from the context that answers the question."
            if stage == "extract"
            else "Copy only the short answer phrase from the context. Do not repeat the context."
        )
        if re.search(r"\bhow many\b", question, re.I):
            instruction = "Answer with only the number requested, using the context."
        context = "\n\n".join(
            f'{item["title"]}\n{item["text"]}' for item in evidence
        )
        return (
            f"{instruction} If the context does not contain the answer, say unknown.\n"
            f"Context:\n{context}\nQuestion: {question}\nAnswer:"
        )

    def pack(self, question: str, evidence: list[dict], stage="answer"):
        """Fairly allocate a fixed input budget, preserving whole words and headers."""
        question = question.strip()
        if not question:
            raise ValueError("Question cannot be empty.")
        words = [re.findall(r"\S+", item["text"]) for item in evidence]

        def make(lengths):
            visible = [
                {key: item[key] for key in ("passage_id", "title", "url")}
                | {"text": " ".join(tokens[:length])}
                for item, tokens, length in zip(evidence, words, lengths)
            ]
            prompt = self.render(question, visible, stage)
            return prompt, visible, self.count_tokens(prompt)

        if not evidence:
            prompt, visible, size = make([])
            if size > self.max_input_tokens:
                raise ValueError("Question exceeds the model input budget.")
            return prompt, visible, size
        _, _, overhead = make([0] * len(words))
        if overhead > self.max_input_tokens - 16:
            raise ValueError("Question/source headers leave too little evidence space; reduce k.")
        low, high = 0, max(map(len, words))
        while low < high:
            middle = (low + high + 1) // 2
            if make([min(middle, len(tokens)) for tokens in words])[2] <= self.max_input_tokens:
                low = middle
            else:
                high = middle - 1
        lengths = [min(low, len(tokens)) for tokens in words]
        # Fill remaining capacity in retrieval order, never truncate the prompt.
        changed = True
        while changed:
            changed = False
            for i, tokens in enumerate(words):
                if lengths[i] < len(tokens):
                    trial = lengths.copy()
                    trial[i] += 1
                    if make(trial)[2] <= self.max_input_tokens:
                        lengths = trial
                        changed = True
        return make(lengths)

    def generate(self, prompt: str) -> dict:
        encoded = self.tokenizer(prompt, return_tensors="pt", truncation=False).to(self.device)
        input_tokens = int(encoded.input_ids.shape[1])
        if input_tokens > self.max_input_tokens:
            raise ValueError(f"Input has {input_tokens} tokens; budget is {self.max_input_tokens}.")
        start = time.perf_counter()
        with self.torch.inference_mode():
            output = self.model.generate(
                **encoded, do_sample=False, num_beams=1,
                max_new_tokens=self.max_new_tokens,
            )
        elapsed = time.perf_counter() - start
        return {
            "text": self.tokenizer.decode(output[0], skip_special_tokens=True).strip(),
            "input_tokens": input_tokens, "output_tokens": int(output.shape[1] - 1),
            "generation_seconds": elapsed,
        }


In [13]:
def supporting_sources(answer: str, visible: list[dict], originals: list[dict]) -> list[dict]:
    if is_unknown(answer):
        return []
    lookup = {item["passage_id"]: item for item in originals}
    sources = []
    for item in visible:
        original = lookup[item["passage_id"]]
        # Check both the model-visible excerpt and the original passage.
        if span(item["text"], answer) is not None:
            quote = span(original["text"], answer)
            if quote is not None:
                sources.append({
                    "passage_id": item["passage_id"], "title": original["title"],
                    "url": original["url"], "answer_span": quote,
                    "visible_excerpt": item["text"],
                    "char_start": original["char_start"], "char_end": original["char_end"],
                })
    return sources

def evidence_sentence(text: str, candidate: str) -> str:
    """Keep the candidate's original sentence so the reduce stage retains its relation."""
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        if span(sentence, candidate) is not None:
            return sentence.strip()
    # A generated span can cross sentence boundaries; retain the visible passage.
    return text


In [14]:
def answer_question(generator: Generator, question: str, retrieved: list[dict],
                    method="concatenate") -> dict:
    if method not in {"concatenate", "generate_from_evidence", "closed_book"}:
        raise ValueError(f"Unknown method: {method}")
    start = time.perf_counter()
    trace, map_steps = [], []
    if method == "closed_book":
        prompt, visible, _ = generator.pack(question, [], stage="closed_book")
        generated = generator.generate(prompt)
        trace.append({"stage": "closed_book", "prompt": prompt, **generated})
        raw_answer = generated["text"]
        sources = []
        answer = UNKNOWN if is_unknown(raw_answer) else raw_answer
    elif not retrieved:
        raw_answer, answer, visible, sources = "", UNKNOWN, [], []
    else:
        evidence = retrieved
        if method == "generate_from_evidence":
            evidence = []
            for passage in retrieved:
                prompt, map_visible, _ = generator.pack(question, [passage], stage="extract")
                generated = generator.generate(prompt)
                trace.append({"stage": "extract", "prompt": prompt, **generated})
                candidate = generated["text"]
                supported = supporting_sources(candidate, map_visible, [passage])
                map_steps.append({
                    "passage_id": passage["passage_id"], "candidate": candidate,
                    "accepted": bool(supported), "visible_evidence": map_visible,
                })
                if supported:
                    # The reduce stage receives original quotes, never unverified model claims.
                    quote = evidence_sentence(map_visible[0]["text"], candidate)
                    map_steps[-1]["evidence_sentence"] = quote
                    evidence.append(passage | {"text": quote})
        if evidence:
            prompt, visible, _ = generator.pack(question, evidence)
            generated = generator.generate(prompt)
            trace.append({"stage": "answer", "prompt": prompt, **generated})
            raw_answer = generated["text"]
            sources = supporting_sources(raw_answer, visible, retrieved)
            # Short factual answers only; never return a generated paragraph as an answer.
            if len(WORD.findall(raw_answer)) > 16:
                sources = []
            if re.search(r"\bhow many\b",question,re.I) and not re.fullmatch(
                r"[\d,]+(?:\s+[A-Za-z]+){0,3}|(?:one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)",raw_answer.strip(),re.I):
                sources = []
            count_question = re.search(r'how many\s+(.+?)(?:\s+(?:are|does|do|in|has|have|exist)\b|\?)',question.casefold()+'?')
            if count_question and sources:
                number = re.match(r'[\d,]+',raw_answer.strip())
                if number:
                    pattern = r'\b'+re.escape(number.group())+r'\s+'+re.escape(count_question.group(1))+r'\b'
                    sources = [source for source in sources if re.search(pattern,source['visible_excerpt'],re.I)]
            answer = raw_answer if sources else UNKNOWN
        else:
            raw_answer, answer, visible, sources = "", UNKNOWN, [], []
    return {
        "question": question, "method": method, "requested_passages_used": len(retrieved),
        "answer": answer, "raw_answer": raw_answer, "abstained": answer == UNKNOWN,
        "span_supported": bool(sources), "citations": sources, "retrieved": retrieved,
        "visible_evidence": visible, "map_steps": map_steps, "trace": trace,
        "generation_calls": len(trace),
        "total_input_tokens": sum(item["input_tokens"] for item in trace),
        "total_output_tokens": sum(item["output_tokens"] for item in trace),
        "latency_seconds": time.perf_counter() - start,
        "grounding_policy": "contiguous lexical span in visible and original evidence"
        if method != "closed_book" else "none (closed-book baseline)",
    }


In [15]:
def focus_context(question,hits,k=3):
    """Lightweight context compression for a small generator; not an entailment verifier."""
    terms={t.casefold().rstrip('s') for t in WORD.findall(question) if t.casefold() not in STOPWORDS|{'there','current','currently','total'}}
    historical=bool(re.search(r'\b(was|were|when|born|founded|died|historical|history|before|during|until|\d{4})\b',question,re.I))
    count=re.search(r'how many\s+(.+?)(?:\s+(?:are|does|do|in|has|have|exist)\b|\?)',question.casefold()+'?')
    ranked=[]
    for hit in hits:
        for match in re.finditer(r'[^\n.!?]+(?:[.!?](?=\s|$)|$)',hit['text']):
            sentence=match.group().strip()
            if len(WORD.findall(sentence))<4:continue
            words={w.casefold().rstrip('s') for w in WORD.findall(sentence)}
            score=6*len(terms&words)/max(1,len(terms))+2*hit['similarity']
            if re.search(r'\b(is|are|has|have|comprising|consists|contains)\b',sentence,re.I):score+=1
            if not historical and re.search(r'\b(was|were|former|formerly|until|before|history|british rule)\b',sentence,re.I):score-=7
            if re.search(r'\b(references|external links|see also|list of)\b',sentence,re.I):score-=5
            if count:
                if re.search(r'\b\d[\d,]*\s+'+re.escape(count.group(1))+r'\b',sentence,re.I):score+=6
                else:score-=5
            left=hit['char_start']+match.start()+len(match.group())-len(match.group().lstrip())
            right=hit['char_start']+match.end()-len(match.group())+len(match.group().rstrip())
            ranked.append((score,dict(hit,text=sentence,original_text=hit['text'],char_start=left,char_end=right)))
    ranked.sort(key=lambda x:x[0],reverse=True)
    selected=[];seen=set()
    for score,p in ranked:
        if score<max(1,ranked[0][0]-4):continue
        signature=' '.join(WORD.findall(p['text'].casefold()))
        if signature in seen or any(p['passage_id']==x['passage_id'] for x in selected):continue
        seen.add(signature);selected.append(p)
        if len(selected)==k:break
    return selected

generator=Generator(MODEL_DIR,device='cpu',threads=QUERY_THREADS,max_new_tokens=24)
def ask_pg(question,k=3,method='concatenate',use_cache=True):
    hits=retriever.search(question,k,use_cache=use_cache)
    timings=dict(retriever.last_timings)
    context_started=time.perf_counter()
    focused=focus_context(question,hits,k)
    timings['context_selection_ms']=(time.perf_counter()-context_started)*1000
    result=answer_question(generator,question,focused,method)
    result['context_passages']=focused
    result['retrieved']=hits
    result['timings']=timings
    result['scope']=SCOPE
    result['schema']=SCHEMA
    result['timings']['answer_ms']=result['latency_seconds']*1000
    return result


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
import ipywidgets as widgets
from IPython.display import display,HTML
import html
from urllib.parse import urlparse
if 'pg_panel' in globals():pg_panel.close()
pg_question=widgets.Textarea(description='Question:',placeholder='What is the capital of India?',layout=widgets.Layout(width='95%',height='80px'))
pg_k=widgets.Dropdown(options=[1,3,5],value=3,description='Passages:')
pg_method=widgets.Dropdown(options=[('Concatenate','concatenate'),('Generate from evidence','generate_from_evidence')],description='Fusion:')
pg_ask=widgets.Button(description='Ask Wikipedia',button_style='primary')
pg_status=widgets.HTML(value=f'Ready — <b>{SCOPE}</b> collection; {expected_articles:,} articles.')
pg_output=widgets.Output()
last_pg_result=None
def submit_pg(_):
    global last_pg_result
    if not pg_question.value.strip():pg_status.value='Enter a question.';return
    controls=[pg_question,pg_k,pg_method,pg_ask]
    for control in controls:control.disabled=True
    pg_output.clear_output(wait=True);pg_status.value='Retrieving and generating…'
    try:
        r=ask_pg(pg_question.value,pg_k.value,pg_method.value);last_pg_result=r
        with pg_output:
            display(HTML('<h3>Answer</h3><p>'+html.escape(r['answer'])+'</p>'))
            print('Coverage:',SCOPE,'| Query cache hit:',r['timings']['cache_hit'])
            for key,value in r['timings'].items():
                if key.endswith('_ms'):print(key+':',round(value,2),'ms')
            for source in r['citations']:
                url=source['url'] if urlparse(source['url']).scheme in {'http','https'} else '#'
                display(HTML('<h4>Source evidence</h4><a href="'+html.escape(url,quote=True)+'" target="_blank" rel="noopener noreferrer">'+html.escape(source['title'])+'</a><blockquote>'+html.escape(source['visible_excerpt'])+'</blockquote>'))
            details=widgets.Output()
            with details:
                for hit in r['retrieved']:print(hit['title'],round(hit['similarity'],3),'\n',hit['text'],'\n')
            accordion=widgets.Accordion(children=[details]);accordion.set_title(0,'All retrieved passages');accordion.selected_index=None;display(accordion)
        pg_status.value='Complete — '+SCOPE+' collection. Source checks are not a correctness guarantee.'
    except Exception as exc:
        last_pg_result=None;pg_status.value='Request failed.'
        with pg_output:print(type(exc).__name__+': '+str(exc))
    finally:
        for control in controls:control.disabled=False
pg_ask.on_click(submit_pg)
pg_panel=widgets.VBox([widgets.HTML('<h2>Wikipedia · pgvector RAG</h2>'),pg_question,widgets.HBox([pg_k,pg_method]),pg_ask,pg_status,pg_output])
display(pg_panel)


## Evaluate latency, recall, and answers separately
The 24 answerable inherited questions are a development set, not a representative Wikipedia benchmark. Pilot articles deliberately include their sources; this makes setup reproducible but biases retrieval difficulty. Add a separate held-out set sampled across all shards, including ambiguous, temporal, paraphrased and truly unanswerable questions before deployment. Do not tune and report on the same questions as an unbiased estimate.

Below: disable application caching, measure each stage, compare ANN top-k with exact half-vector top-k, and check annotated evidence recall. Exact search is gated to small collections because full-corpus scans per question would be expensive. The report distinguishes unavailable full-scale recall from zero recall. For full scale, use a separately bounded representative evaluation collection and inspect EXPLAIN/BUFFERS. Latencies are single-client warm local measurements, not load-test SLAs; report p50/p95/p99 and recall together.


In [17]:
def benchmark_pg(questions,repeats=3,exact_row_limit=100000):
    rows=[]
    count=store.meta('audit')['chunks']
    for q in questions:
        exact=None
        if count<=exact_row_limit:
            v=HalfVector(query_embedder.encode([q['question']],query=True)[0])
            with retriever.db.transaction():
                retriever.db.execute('SET LOCAL enable_indexscan=off')
                retriever.db.execute('SET LOCAL enable_bitmapscan=off')
                exact={x['id'] for x in retriever.db.execute(f'SELECT id FROM {SCHEMA}.chunks ORDER BY embedding <=> %s::halfvec(384),id LIMIT 5',(v,)).fetchall()}
        for repeat in range(repeats):
            hits=retriever.search(q['question'],5,use_cache=False)
            rows.append(dict(question_id=q['id'],repeat=repeat,
                ann_recall_at_5=len(exact&{h['id'] for h in hits})/len(exact) if exact else None,
                gold_hit_at_5=any(h['article_id']==q['article_id'] and q['evidence_quote'].casefold() in h['text'].casefold() for h in hits),
                **retriever.last_timings))
    return rows

if RUN_EVALUATION:
    questions=[q for q in DEVELOPMENT_QUESTIONS if q['answerable']]
    measurements=benchmark_pg(questions)
    summary={key:{f'p{p}':float(np.percentile([r[key] for r in measurements],p)) for p in [50,95,99]}
        for key in ['query_embedding_ms','ann_rerank_ms','fetch_ms','total_ms']}
    run_dir=ROOT/'results'/('pgvector_'+SCOPE+'_'+time.strftime('%Y%m%d_%H%M%S'));run_dir.mkdir(parents=True,exist_ok=True)
    (run_dir/'retrieval.json').write_text(json.dumps(measurements,indent=2))
    (run_dir/'summary.json').write_text(json.dumps(summary,indent=2))
    (run_dir/'manifest.json').write_text(json.dumps(dict(config=config,index_kind=INDEX_KIND,
        candidates=ANN_CANDIDATES,ef_search=EF_SEARCH,query_threads=QUERY_THREADS,scope=SCOPE,
        corpus=store.meta('audit'),python=sys.version,
        packages={name:importlib.metadata.version(name) for name in ['torch','transformers','sentence-transformers','psycopg','pgvector']},
        plan=plan),indent=2))
    print(json.dumps(summary,indent=2))
    print('Development evidence hit@5:',np.mean([r['gold_hit_at_5'] for r in measurements]))
    print('Saved:',run_dir)
else:print('Evaluation skipped.')


Evaluation skipped.


In [18]:
RUN_ANSWER_ABLATIONS = False  # Deliberately independent from the retrieval benchmark.
if RUN_ANSWER_ABLATIONS:
    from collections import Counter
    import string
    def normalized(s):
        return ' '.join(re.sub(r'\b(a|an|the)\b',' ',s.lower().translate(str.maketrans('','',string.punctuation))).split())
    def answer_scores(answer,aliases):
        a=normalized(answer);em=f1=0.0
        for alias in aliases:
            b=normalized(alias);em=max(em,float(a==b))
            overlap=sum((Counter(a.split()) & Counter(b.split())).values())
            if overlap:f1=max(f1,2*overlap/(len(a.split())+len(b.split())))
        return em,f1
    ablations=[]
    for q in [q for q in DEVELOPMENT_QUESTIONS if q['answerable']]:
        raw=retriever.search(q['question'],5,use_cache=False)
        shared_retrieval=dict(retriever.last_timings)
        for method,k in [('closed_book',0)]+[(m,k) for m in ['concatenate','generate_from_evidence'] for k in [1,3,5]]:
            context=focus_context(q['question'],raw[:k],k) if k else []
            result=answer_question(generator,q['question'],context,method)
            em,f1=answer_scores(result['answer'],q['answers'])
            ablations.append(dict(id=q['id'],method=method,k=k,em=em,f1=f1,
                actual_context_count=len(context),shared_retrieval_timings=shared_retrieval if k else None,result=result))
    output=ROOT/'results'/('pgvector_answers_'+SCOPE+'_'+time.strftime('%Y%m%d_%H%M%S'))
    output.mkdir(parents=True,exist_ok=True)
    (output/'ablations.json').write_text(json.dumps(ablations,indent=2))
    print('Saved answer ablations:',output)


## Full-corpus operating notes and limits
- Full mode streams all 6,407,814 articles from the pinned 20231101.en snapshot. The new token chunker produces a different passage count from the old 42.5-million-passage SQLite index. It does not reuse incompatible old vectors.
- Existing SQLite files and notebooks are read only. PostgreSQL schemas isolate pilot and full collections. Installing pgvector alone does not populate either collection.
- Query-time caching is bounded and per kernel. Cache hits exclude fresh query embedding and should never be used to claim new-query latency.
- A 384-dimensional half vector needs about 768 bytes of numeric payload, plus row metadata, text, indexes and WAL. Binary signatures need 48 numeric bytes, but the HNSW graph adds considerable overhead. Full-scale memory/disk and p95 latency must be measured; the pilot does not establish a 10 ms guarantee.
- Full text search is not added by default because the existing broad lexical search was the latency bottleneck. If semantic recall on names/codes is weak, evaluate a separately indexed lexical path and reciprocal rank fusion; do not silently add a global scan to every query.
- Production deployment still needs least-privilege roles, authentication/TLS as appropriate, backups and restore tests, collection lifecycle management, monitoring, concurrency/load tests, and independent answer-quality/faithfulness evaluation. Never treat source-span checks as entailment or BGE similarity as calibrated confidence.
- Close the UI connections at the end with `retriever.close(); store.close()`. Do not execute that while using the UI.

Sources: [BGE model card](https://huggingface.co/BAAI/bge-small-en-v1.5), [pgvector indexing and bulk loading](https://github.com/pgvector/pgvector), [Sentence Transformers performance](https://www.sbert.net/docs/sentence_transformer/usage/efficiency.html). Wikipedia text is attributed through retained article URLs; see [dataset licensing](https://huggingface.co/datasets/wikimedia/wikipedia). The RAG session is reference material; its illustrative product examples and installation cells were not executed.
